# Assignment 5
## Data Preprocessing

We will be using a dataframe created from *Income Dirty Data.csv*. Download the file from D2L. 

1. Import the following modules
    - `pandas`
    - `numpy`
    - `preprocessing` from `sklearn` (for bonus question)
    - `KNNImputer` from `sklearn.impute` (for bonus question)
2. Create your dataframe from the file using `pandas`

In [15]:
# Code here
import pandas as pd
import numpy as np
from sklearn import preprocessing
from sklearn.impute import KNNImputer

df = pd.read_csv("Income Dirty Data.csv")
print(df.head())

   ID     sex  age    income  tax_15_pct
0   1   Women   21  147168.0    22075.20
1   2  Female   29  119595.0    17939.25
2   3  Female   56   87770.0    13165.50
3   4     NaN   21   54259.0     8138.85
4   5    Male   28       NaN   160230.00


3. Calculate and display the following information
    - Total number of NaN values for **each column**
    - Percentage of NaN values in the dataset 
    - Number of rows *without* any NaN values

In [3]:
# Code here
nan_per_column = df.isna().sum()
print("NaN values per column: ")
print(nan_per_column)

percent_nan = df.isna().mean().mean() * 100
print(f"\nPercent of NaN values in the dataset: {percent_nan:.2f}%")

rows_without_nan = df.dropna().shape[0]
print(f"\nNumber of rows without any NaN values: {rows_without_nan}")

NaN values per column: 
ID              0
sex            88
age             0
income        109
tax_15_pct     93
dtype: int64

Percent of NaN values in the dataset: 5.80%

Number of rows without any NaN values: 733


Besides missing values (NaN), the dataset contains errors. We have the following rules to check:

* All employees are adults (18+ years old)
* All employees pay 15% of their income for the tax
* All employees make money; no income should be <= 0 
---
4. Calculate and display the percentage of the data that does **NOT** violate any **one** of the rules.

In [6]:
# Code here
valid_mask = (
    (df["age"]        >= 18)  &
    (df["income"]     > 0)    &
    (df["tax_15_pct"] > 0)    &
    (df["tax_15_pct"] == df["income"] * 0.15)
)

valid_percentage = valid_mask.mean() * 100
print(f"Percentage of data that does not violate any rule: {valid_percentage}%")

Percentage of data that does not violate any rule: 47.0%


Now that we have determined the number of erroneous datapoints in our set, let's work on correcting it as best we can.

5. Replace non *Female*/*Male* values in the **Sex** column with either *Female* or *Male* (e.g., Women --> Female)

In [20]:
# Code here
df["sex"] = df["sex"].astype(str).str.strip().str.lower()
df["sex"] = df["sex"].replace({
    "women": "female",
    "woman": "female",
    "men": "male",
    "man": "male"
})
df["sex"] = df["sex"].where(df["sex"].isin(["female", "male"]), np.nan)

print(df["sex"].value_counts(dropna=False))

sex
male      457
female    455
NaN        88
Name: count, dtype: int64


6. Replace non-positive **Age** values with NaN (`numpy.NaN`)
7. Replace non-positive **Income** values with NaN (`numpy.NaN`)
8. Replace non-positive **Tax (15%)** values with NaN (`numpy.NaN`)

In [13]:
# Code here
df["age"] = df["age"].where(df["age"] > 0, np.nan)
df["income"] = df["income"].where(df["income"] > 0,np.nan)
df["tax_15_pct"] = df["tax_15_pct"].where(
    df["tax_15_pct"] > 0,
    np.nan
)

The following question is a bonus (+10) question, but I'd encourage you to give it a try!

9. Use machine learning (`KNNImputer`) to impute all missing values (replaces NaN values with the most predicted values)
    - Will need to use a scaler and convert the values in the **Sex** column to a numeric value for algorithm to work properly
    - Show some of the data prior to imputing, and after imputing

In [22]:
# Code here
print("Before imputation:")
print(df[["sex", "age", "income", "tax_15_pct"]].head(10))

sex_map = {"female": 0, "male": 1}
df_numeric = df.copy()
df_numeric["sex"] = df_numeric["sex"].map(sex_map)

imputer = KNNImputer(n_neighbors=5)
imputed_data = imputer.fit_transform(df_numeric[["sex", "age", "income", "tax_15_pct"]])

imputed_df = pd.DataFrame(
    imputed_data,
    columns=["sex", "age", "income", "tax_15_pct"],
    index=df_numeric.index
)

scaler = preprocessing.StandardScaler()
scaled_data = scaler.fit_transform(imputed_df)

scaled_back = pd.DataFrame(
    scaler.inverse_transform(scaled_data),
    columns=imputed_df.columns,
    index=imputed_df.index
)

scaled_back["sex"] = scaled_back["sex"].round().clip(0, 1).map({0: "female", 1: "male"})

print("\nAfter imputation:")
print(scaled_back[["sex", "age", "income", "tax_15_pct"]].head(10))

Before imputation:
      sex  age    income  tax_15_pct
0  female   21  147168.0    22075.20
1  female   29  119595.0    17939.25
2  female   56   87770.0    13165.50
3     NaN   21   54259.0     8138.85
4    male   28       NaN   160230.00
5  female    0  128326.0    19248.90
6  female   -1       NaN         NaN
7  female   24       NaN    11820.60
8  female   38  149473.0    22420.95
9    male   48  113663.0  1136630.00

After imputation:
      sex   age    income  tax_15_pct
0  female  21.0  147168.0    22075.20
1  female  29.0  119595.0    17939.25
2  female  56.0   87770.0    13165.50
3    male  21.0   54259.0     8138.85
4    male  28.0  123059.4   160230.00
5  female   0.0  128326.0    19248.90
6  female  -1.0   80854.2    65575.86
7  female  24.0  124442.0    11820.60
8  female  38.0  149473.0    22420.95
9    male  48.0  113663.0  1136630.00


10. In the empty `Markdown` cell below, explain why it is important to clean a dataset before calculating analytics about the data

It is important to clean a dataset before calculating analytics because dirty data can distort the results. Missing values, incorrect labels, and invalid numbers can make averages, percentages, and trends misleading. If bad data stays in the dataset, the analysis can lead to incorrect conclusions. Cleaning the data helps ensure the results are accurate, trustworthy, and based on real information.

### Submission to D2L Dropbox
- Submit this `Jupyter` file to D2L, renamed as **Last_First_Assignment5.ipynb** 
    - Replace '**Last**' and '**First**' with your first and last name

- Include the link to your GitHub repository (the URL of your repo page, for
   example `https://github.com/yourname/csci-4047-work`).
### How to add my code to GitHub?

1. Stage your file (this tells Git which changes to include):

      `git add Last_First_Assignment5.ipynb`

   To stage everything (you might not want to stage everything though) in the folder instead, use `git add .`

3. Commit your changes (this saves a snapshot with a message):

       git commit -m "Add Assignment 5"

   The text in quotes is your commit message. Make it describe what you
   did.

4. Push your commit up to GitHub:

       git push -u origin main

   The `-u origin main` part is only needed the first push. After that,
   `git push` alone is enough.

**What each command does, briefly**

- `git add` picks which files to include in the next save.
- `git commit` saves a snapshot of those files on your computer, with a
  message describing the change.
- `git push` uploads your saved commits to GitHub so they appear online.